In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py

from tqdm import tqdm

from glob import glob

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_Plotting_CLASS 

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=os.path.join(dir2,'Code/CodeFiles/Functions')
sys.path.append(path)

import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=False

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Np, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
#############################################
#LOADING DATA FUNCTIONS

In [ ]:
def job_filter(arr, start_job,end_job):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
def FilterTrackedArrays(trackedArrays, start_job, end_job):
    filtered = {}
    for category, subDict in trackedArrays.items():
        filtered[category] = {}
        for key, out_arr in subDict.items():
            if out_arr is None or len(out_arr) == 0:
                filtered[category][key] = out_arr
            else:
                filtered[category][key] = job_filter(out_arr, start_job, end_job)
    return filtered

def Get_Lagrangian_Binary_Array_Data(ModelData, start_job,end_job):
    directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
    Lagrangian_Binary_Array_Data,files = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory_Lagrangian_Binary_Array, ModelData, start_job,end_job)

    return Lagrangian_Binary_Array_Data

def GetSpatialData(Lagrangian_Binary_Array_Data):
    Z = Lagrangian_Binary_Array_Data['Z'].data.compute()
    Y = Lagrangian_Binary_Array_Data['Y'].data.compute()
    X = Lagrangian_Binary_Array_Data['X'].data.compute()
    return Z,Y,X

def GetPropertyData(start_job,end_job):
    
    QV,TH_V,HMC = [],[],[]
    for timeString in tqdm(ModelData.timeStrings):
        QV_t = CallLagrangianArray(ModelData, DataManager, timeString, 'QV')[start_job:end_job]
        TH_V_t = CallLagrangianArray(ModelData, DataManager, timeString, 'THETA_v')[start_job:end_job]
        #HMC_t = CallLagrangianArray(ModelData, DataManager, timeString, 'HMC')[start_job:end_job]
        QV.append(QV_t); TH_V.append(TH_V_t); #HMC.append(HMC_t)
        
    QV = np.array(QV);TH_V = np.array(TH_V)#; HMC = np.array(HMC)
    return QV,TH_V#,HMC

def GetParcelData(trackedArrays):
    nonCL = trackedArrays['nonCL'].copy()
    CP = trackedArrays['ColdPool'].copy()
    SBF = trackedArrays['SBF'].copy()
    
    nonCL_all = nonCL['ALL']
    # nonCL_shallow = nonCL['SHALLOW']
    # nonCL_deep = nonCL['DEEP']
    
    CP_all = CP['ALL']
    # CP_shallow = CP['SHALLOW']
    # CP_deep = CP['DEEP']

    SBF_all = SBF['ALL']
    # SBF_shallow = SBF['SHALLOW']
    # SBF_deep = SBF['DEEP']

    
    return nonCL_all, CP_all, SBF_all

In [ ]:
#############################################
#LOADING DATA

In [ ]:
#READING BACK IN SUBSETTED TRACKED PARCEL DATA
trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                         Results_InputOutput_Class)
trackedArraysFiltered = FilterTrackedArrays(trackedArrays, start_job, end_job)

In [ ]:
Lagrangian_Binary_Array_Data = Get_Lagrangian_Binary_Array_Data(ModelData, start_job,end_job)
[Z,Y,X] = GetSpatialData(Lagrangian_Binary_Array_Data)
[QV,TH_V] = GetPropertyData(start_job,end_job)

In [ ]:
def GetParcelData(trackedArrays):
    nonCL = trackedArrays['nonCL'].copy()
    CP = trackedArrays['ColdPool'].copy()
    SBF = trackedArrays['SBF'].copy()
    
    nonCL_all = nonCL['ALL']
    # nonCL_shallow = nonCL['SHALLOW']
    # nonCL_deep = nonCL['DEEP']
    
    CP_all = CP['ALL']
    # CP_shallow = CP['SHALLOW']
    # CP_deep = CP['DEEP']

    SBF_all = SBF['ALL']
    # SBF_shallow = SBF['SHALLOW']
    # SBF_deep = SBF['DEEP']

    
    return nonCL_all, CP_all, SBF_all

In [ ]:
[nonCL_all, CP_all, SBF_all]=GetParcelData(trackedArraysFiltered)

In [ ]:
#############################################
#CALCULATION FUNCTIONS

In [ ]:
def NanDivide(numerator, denominator):
    result = np.full_like(numerator, np.nan, dtype=float)
    
    valid = denominator != 0
    result[valid] = numerator[valid] / denominator[valid]
    
    return result

def ApplyBinning(Ydata,Xdata,data):
    dataSum = np.zeros((ModelData.Nyh, ModelData.Nxh))
    dataCount = np.zeros((ModelData.Nyh, ModelData.Nxh))

    # accumulate
    np.add.at(dataSum, (Ydata, Xdata), data)
    np.add.at(dataCount, (Ydata, Xdata), 1)

    dataMean = NanDivide(dataSum, dataCount)
    return dataMean

def GetDataMean(parcelArray,variableData):
    pList = parcelArray[:,0]
    initialTime = parcelArray[:,1]
    
    data = variableData[initialTime,pList]
    Zdata = Z[initialTime,pList]
    Ydata = Y[initialTime,pList]
    Xdata = X[initialTime,pList]
    #==>
    dataMean = ApplyBinning(Ydata,Xdata,data)
    return dataMean

In [ ]:
#############################################
#CALCULATION

In [ ]:
dataMean_1_all = GetDataMean(parcelArray=nonCL_all,variableData=QV)
dataMean_2_all = GetDataMean(parcelArray=nonCL_all,variableData=TH_V)

dataMean_3_all = GetDataMean(parcelArray=CP_all,variableData=QV)
dataMean_4_all = GetDataMean(parcelArray=CP_all,variableData=TH_V)

dataMean_5_all = GetDataMean(parcelArray=SBF_all,variableData=QV)
dataMean_6_all = GetDataMean(parcelArray=SBF_all,variableData=TH_V)

In [ ]:
#############################################
#PLOTTING FUNCTIONS

In [ ]:
def MakeSinglePlot(ax, dataMean, fig,
                   cmap='viridis',
                   xLabel=True,yLabel=True):
    pc = ax.pcolormesh(dataMean, cmap=cmap); fig.colorbar(pc, ax=ax)
    if xLabel: ax.set_xlabel('X')
    if yLabel: ax.set_ylabel('Y')
    ax.set_xlim(100, 450); ax.set_ylim(75, 125)
    ax.axvline(ocean_location,color='red')
ocean_location = int(np.round((ModelData.xh-ModelData.xh[0])[-1]*0.25,0))

def Plot_XProfile(ax, dataMean_nonCL, dataMean_CP, title, ylabel, color, labels, xLabel=True,
                  xh = ModelData.xh-ModelData.xh[0]):
    
    import warnings
    warnings.filterwarnings("ignore", message="Mean of empty slice")
    
    mean_nonCL = np.nanmean(dataMean_nonCL, axis=0)
    mean_CP    = np.nanmean(dataMean_CP, axis=0)

    ax.plot(xh, mean_nonCL, color='orange', alpha=0.8, linestyle='-',  label=labels[0], zorder=10)
    ax.plot(xh, mean_CP,    color='purple', linestyle='-', label=labels[1],zorder=5)
    ax.set_title(title)
    if xLabel: ax.set_xlabel('X')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

    ax.axvline(ocean_location,color='red')

In [ ]:
#############################################
#PLOTTING

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8),sharex=True,gridspec_kw={"wspace":0.05})

dataList = [dataMean_1_all*1e3, dataMean_2_all, dataMean_3_all*1e3, dataMean_4_all,dataMean_5_all*1e3, dataMean_6_all]
titles = ['nonCL - QV', 'nonCL - TH_V',
          'CP - QV', 'CP - TH_V',
          'SBF - QV', 'SBF - TH_V']
cmaps = ['viridis','viridis','viridis','viridis','viridis','viridis']

for i, (ax, dataMean, title, cmap) in enumerate(tqdm(list(zip(axes.flat, dataList, titles, cmaps)))):
    xLabel = True if i >= 4 else False
    yLabel = True if i % 2 == 0 else False
    MakeSinglePlot(ax, dataMean, fig, cmap, xLabel=xLabel,yLabel=yLabel)
    ax.set_title(title)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

labels = ['nonCL','CP']
Plot_XProfile(axes[0,0], dataMean_1_all*1e3, dataMean_3_all*1e3, 'QV', 'QV', color='k', labels=labels, xLabel=False)
Plot_XProfile(axes[0,1], dataMean_2_all,      dataMean_4_all,      'TH_v', 'TH_v', color='k', labels=labels, xLabel=False)

labels = ['nonCL','SBF']
Plot_XProfile(axes[1,0], dataMean_1_all*1e3, dataMean_5_all*1e3, 'QV', 'QV', color='k', labels=labels)
Plot_XProfile(axes[1,1], dataMean_2_all,      dataMean_6_all,      'TH_v', 'TH_v', color='k', labels=labels)

#==> thermals are selectively moister than, but less buoyant than CLs
#==> thermals are selecting moist, though lower th_v parcels, while SBF 
#==> CP and SBF (CL) are less selective, resulting in mixure of air-masses with environmental air ==>